## Bài 1: Word Frequency Dictionary ##

Yêu cầu: Viết hàm tính tần suất xuất hiện của các từ từ danh sách document.
- Input: <br>
D1: the quick brown fox <br>
D2: the lazy dog sleeps <br>
D3: the fox jumps over the dog <br>
- Output mong muốn:
{
  "the": 4,
  "fox": 2,
  "dog": 2,
  "quick": 1,
  "brown": 1,
  "lazy": 1,
  "sleeps": 1,
  "jumps": 1,
  "over": 1
}

In [1]:
def word_frequency(docs):
    freq={}
    for doc in docs:
        words = doc.split()
        for word in words:
            if word in freq:
                freq[word]+=1
            else:
                freq[word]=1
    return freq
docs = [
    "the quick brown fox",
    "the lazy dog sleeps",
    "the fox jumps over the dog"
]
print(word_frequency(docs))

{'the': 4, 'quick': 1, 'brown': 1, 'fox': 2, 'lazy': 1, 'dog': 2, 'sleeps': 1, 'jumps': 1, 'over': 1}


## Bài 2: Text Chunking ##

Yêu cầu: Chia danh sách tokens thành các đoạn nhỏ (chunks) với kích thước cố định và có thể overlap.

- VD: 
tokens = ["a", "b", "c", "d", "e", "f"]
chunk_size = 3
overlap = 0
- Output:
[["a", "b", "c"], ["d", "e", "f"]]


In [2]:
def chunk_tokens(tokens, chunk_size,overlap):
    chunks = []
    step = chunk_size - overlap
    
    for i in range(0,len(tokens),step):
        chunk = tokens[i:i+chunk_size]
        if len(chunk) == chunk_size:
            chunks.append(chunk)
    return chunks

In [3]:
tokens = ["a", "b", "c", "d", "e", "f"]
print (chunk_tokens(tokens,3,1))

[['a', 'b', 'c'], ['c', 'd', 'e']]


## Bài 3: Web Crawler ##

Yêu cầu: Implement một hàm crawl toàn bộ URL từ một trang web bắt đầu (entrypoint).

In [ ]:
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse
from collections import deque
import requests


def crawl(
    url: str,
    strategy: str = "BFS",  
    max_depth: int = 2,
    max_pages: int = 50,
    include_external: bool = False
):
    visited = set()
    results = []

    # xác định domain gốc
    base_domain = urlparse(url).netloc

    # chọn cấu trúc dữ liệu
    if strategy == "BFS":
        container = deque()
        push = container.append
        pop = container.popleft
    else:  # DFS
        container = []
        push = container.append
        pop = container.pop

    # (url, depth)
    push((url, 0))
    visited.add(url)

    while container and len(results) < max_pages:
        current_url, depth = pop()
        results.append(current_url)

        if depth >= max_depth:
            continue

        try:
            response = requests.get(current_url, timeout=5)
            if "text/html" not in response.headers.get("Content-Type", ""):
                continue

            soup = BeautifulSoup(response.text, "html.parser")

            for tag in soup.find_all("a", href=True):
                href = tag["href"]

                # chuyển relative → absolute
                next_url = urljoin(current_url, href)

                # chuẩn hóa (bỏ fragment #)
                next_url = next_url.split("#")[0]

                # check domain
                domain = urlparse(next_url).netloc
                if not include_external and domain != base_domain:
                    continue

                if next_url not in visited:
                    visited.add(next_url)
                    push((next_url, depth + 1))

        except Exception:
            continue

    return results

In [5]:
urls = crawl(
    "https://www.dienmayxanh.com/",
    strategy="BFS",
    max_depth=1,
    max_pages=10
)

for u in urls:
    print(u)

https://www.dienmayxanh.com/


## Bài 4. Giả sử trong quá trình gọi API từ LLM model có thể xảy ra lỗi (có thể timeout, rate limit, lỗi 5XX) ##

Xây dựng cơ chế retry khi xuất hiện error. Ví dụ

- TimeoutError
- ConnectionError

Yêu cầu có thể config params:

- max_retries
- delay
- backoff: tăng dần thời gian chờ giữa các lần retry, delay = base_delay × (backoff ^ attempt)

In [6]:
import time
import random

def call_api(url):
    r = random.random()
    if r < 0.5:
        raise TimeoutError("Timeout")
    return {"status": 200, "data": "OK"}


def call_with_retry(
    url,
    max_retries=3,
    base_delay=1,
    backoff=2
):
    attempt = 0

    while attempt <= max_retries:
        try:
            result = call_api(url)
            print(f"Success at attempt {attempt}")
            return result

        except (TimeoutError, ConnectionError) as e:
            if attempt == max_retries:
                print("Hết retry")
                raise e

            delay = base_delay * (backoff ** attempt)
            print(f"Retry {attempt + 1} after {delay}s due to {e}")

            time.sleep(delay)
            attempt += 1

In [7]:
print(call_with_retry("https://www.dienmayxanh.com/"))

Retry 1 after 1s due to Timeout
Success at attempt 1
{'status': 200, 'data': 'OK'}
